# 3. Heterogeneity Computations
The goal in this notebook is to generate estimates of heterogeneity by computing $I^2$, the proportion between-study variance due to heterogeneity rather than random sampling error. We compute this via simulation. Should be run after the Stan model is run, but before the visuals are generated.

In [8]:
library(tidyverse)
library(ggplot2)
library(rstan)
library(Hmisc)
set.seed(20)

inv_logit = plogis
logit = qlogis

In [9]:
base_dir = getwd()
io_set = "Main Results"

model_set = "LogisticRegression"
model_input_dir  = file.path(base_dir, "Processed Data", io_set)
model_output_dir = file.path(base_dir, "Model Output", io_set)
output_dir = file.path(base_dir, "Heterogeneity Estimates", io_set)

sev_class_types = c("1997type", "2009type", "hospitalisation")
#sev_class_type = "1997type"
#sev_class_type = "2009type"
sev_class_type = "hospitalisation"
save_output = TRUE

In [10]:
#Read in the model input data (which gives information on outcomes and scenarios)
curr_sev_class_type = sev_class_types[[1]]
res_suffix = ""
generate_I2_estimates = function(curr_sev_class_type, res_suffix){
    #Read input data used to generate results
    input_data = file.path(model_input_dir, paste0("data_", curr_sev_class_type, res_suffix, ".rds")) %>% readRDS

    #Read in the model results. We are primarily interested in the pooled estimates p and the outcome estimates theta
    model_results = file.path(model_output_dir, paste0("Results_", model_set, "_mean=0_sd=2_sd_mean=0.5_sdsd=2_", curr_sev_class_type, ".rds")) %>% readRDS

    #Extract some variables from input data to get correct indices on things
    scenario_df = input_data$scenario_df
    outcome_df = input_data$outcome_df %>% mutate(ThetaIndex = 1:nrow(.))

    #We only generate I^2 estimates for included scenarios (those with at least 2 studies contributing data)
    included_scenarios = scenario_df %>% filter(Inclusion == "Included")

    #Get the available posterior samples for p and theta
    p_est_draws = extract(model_results, pars = "p")$p
    theta_est_draws = extract(model_results, pars = "theta")$theta
    num_samples = nrow(p_est_draws) # Number of posterior samples
    num_included_scenarios = nrow(included_scenarios) #Number of included scenarios to generate I^2 estimates for 
    I_2_est_mat = array(-1, dim = c(num_included_scenarios, num_samples)) #Matrix of I^2 estimates (with each posterior sample corresponding to one I^2 estimate)
    correction = 0.00001 #To prevent NaN in I2 estimates

    #For each row corresponding to an included scenario
    for(curr_row_ind in 1:num_included_scenarios){
        curr_row = included_scenarios[curr_row_ind, ]
        curr_scenario = curr_row$Scenario #Name of scenario to retrieve from outcome_df
        curr_scenario_ind = curr_row$RegSeroPriorInd #Index of Scenario so we know which value of p to get
        curr_outcomes = outcome_df %>% filter(Scenario == curr_scenario) #Get the relevant outcomes that are part of the scenario
        num_outcomes = nrow(curr_outcomes) #Get number of outcomes
        curr_p_vals = p_est_draws[ , curr_scenario_ind] #Get the values of p (column corresponding to the scenario index)
        
        curr_theta_indices = curr_outcomes %>% pull(ThetaIndex) #Get the theta index of each outcome. These should just be from 1:num_outcomes in the order of the outcome_df
        curr_n_vals = curr_outcomes %>% pull(N) #Sample size of each outcome in the given scenario
        curr_theta_vals = theta_est_draws[, curr_theta_indices] #matrix of theta values that match the outcomes for the given scenario
        
        #For each sample, we compute an estimate of I^2 (this means matching indices of p and thetas)
        I_2_est_samples = array(-1, dim = num_samples)
        for(curr_sample_ind in 1:num_samples){
            curr_p = curr_p_vals[curr_sample_ind] #Current p sample from the posterior distribution
            curr_theta = curr_theta_vals[curr_sample_ind, ] #Current theta samples from the posterior distribution
            
            without_het = array(-1, dim = length(num_outcomes)) #Between study variance in the absence of heterogeneity
            with_het = array(-1, dim = length(num_outcomes)) #Between study variance given heterogeneity
            
            for(i in 1:num_outcomes){
                without_het[i] = rbinom(n = 1, size = curr_n_vals[i], prob = curr_p) #For each outcome in the scenario, generate 1 random binomial draw using the outcome's sample size and the pooled estimate

                #We also generate 1 random binomial draw still using the outcome's sample size,
                #but this time using theta as the probability (includes random effect)
                with_het[i] = rbinom(n  = 1, size = curr_n_vals[i], prob = curr_theta[i]) 
            }

            #Convert the binomial draws to proportions by dividing by the sample sizes
            without_het_props = without_het / curr_n_vals
            with_het_props = with_het / curr_n_vals

            #Compute the variance between proportions with and without heterogeneity - adding in the correction to both values to prevent divisions by 0 
            #which could occur for scenarios with only a small number of studies and/or smalller sample sizes
            var_het = var(with_het_props) + correction
            var_without_het = var(without_het_props) + correction
            I_2_est_samples[curr_sample_ind] = (var_het - var_without_het) / var_het
        }
        #Set the I^2 estimates for the scenario to the samples computed 
        I_2_est_mat[curr_row_ind, ] = I_2_est_samples
    }
    #Compute the I^2 estimate for each scenario by taking the median. We replace any values < 0 with 0
    #These negative values can occur in scenarios with a small number of studies and/or small samples sizes or those with barely any heterogeneity. 
    median_I2 = apply(I_2_est_mat, 1, median)
    median_I2[median_I2 < 0] = 0 #Replace negative values with 0 

    scenario_I2 = included_scenarios %>% mutate(I2_est = median_I2) 
    output = scenario_df %>% left_join(scenario_I2 %>% select(Scenario, I2_est), by = "Scenario")#Create a DataFrame with scenario labels and the I^2 estimates to be output
    return(output)
}

In [11]:
res_suffix = ""
# orig_1997 = generate_I2_estimates(sev_class_types[[1]], res_suffix)
# orig_2009 = generate_I2_estimates(sev_class_types[[2]], res_suffix)
# orig_hosp = generate_I2_estimates(sev_class_types[[3]], res_suffix)

for(curr_sev_class_type in sev_class_types){
    output = generate_I2_estimates(curr_sev_class_type, res_suffix)
    if(save_output){
        saveRDS(output, file.path(output_dir, paste0("I2_Estimates_", curr_sev_class_type, res_suffix, ".rds"))) #Save the output to RDS file for use in visualisations
    }
}
